In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/minidata/data.npy
/kaggle/input/alldata/data.npy
/kaggle/input/vector/df_iemocap.csv
/kaggle/input/vector/audio_vectors_2.pkl
/kaggle/input/vector/audio_vectors_5.pkl
/kaggle/input/vector/audio_vectors_1.pkl
/kaggle/input/vector/audio_vectors_3.pkl
/kaggle/input/vector/audio_vectors_4.pkl
/kaggle/input/datafixed/data.npy
/kaggle/input/autoencoders/Dautoencoder.pkl


In [2]:
import torch
import pandas as pd
import torch.nn as nn
from torch.autograd import Variable
from tqdm import tqdm
#import hiddenlayer as hl

In [3]:
class AutoEncoder(nn.Module):
    def __init__(self):
        super(AutoEncoder, self).__init__()
        self.Encoder = nn.Sequential(
            # param [input_c, output_c, kernel_size, stride, padding]
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=9, stride=2, padding=4),
            nn.LeakyReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=7, stride=2, padding=3),
            nn.LeakyReLU(),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=5, stride=2, padding=2),
            nn.LeakyReLU(),
            nn.Flatten(),
            nn.Linear(8192, 256)
        )
        self.Decoder_part1 = nn.Sequential(
            nn.Linear(256, 8192),
            nn.LeakyReLU(),
        )
        self.Decoder_part2 = nn.Sequential(
            #in_c, out_c, kernel, stride, padding
            nn.ConvTranspose2d(128, 128, 5, 2, 2, output_padding=1),
            nn.LeakyReLU(),
            nn.ConvTranspose2d(128, 64, 7, 2, 3, output_padding=1),
            nn.LeakyReLU(),
            nn.ConvTranspose2d(64, 32, 9, 2, 4, output_padding=1),
            nn.LeakyReLU(),
            nn.Conv2d(32, 1, 1, 1, 0),
            nn.Sigmoid()
        )
    def forward(self, x):
        encoded = torch.zeros(5,3)
        encoded.to(device)
        encoded = self.Encoder(x)
        x = self.Decoder_part1(encoded)
        x = torch.reshape(x, (-1, 128, 8, 8))
        x = self.Decoder_part2(x)
        return encoded, x.float()

In [4]:
data = np.load("/kaggle/input/alldata/data.npy")
EPOCH = 3
data[0].shape

(64, 64)

In [5]:
print(data[3])

[[7.07190394e-01 5.12105584e-01 7.54191816e-01 ... 6.70001090e-01
  7.90788889e-01 5.66662133e-01]
 [6.77332878e-01 6.51195884e-01 7.49762952e-01 ... 6.17066681e-01
  7.87475049e-01 4.96427655e-01]
 [2.05004320e-01 2.16610849e-01 3.66785705e-01 ... 2.30955660e-01
  2.95220017e-01 1.33331954e-01]
 ...
 [1.39039611e-16 2.86838410e-17 9.16515854e-17 ... 6.00606777e-17
  6.58816469e-17 3.49795557e-17]
 [5.89495888e-17 2.62271113e-17 5.17007338e-17 ... 3.51746710e-17
  5.79081486e-17 2.93635353e-17]
 [3.91718791e-17 2.97188723e-17 5.09851895e-17 ... 2.82635373e-17
  4.56152488e-17 2.79350606e-17]]


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [7]:
def add_noise(inputs,noise_factor=0.3):
    noisy = inputs + (torch.randn_like(inputs) * noise_factor)
    noisy = torch.clip(noisy,0.,1.)
    return noisy

In [8]:
from torch.utils.data import Dataset, DataLoader
class Mydataset(Dataset):
    def __init__(self, training):
        self.data = training
    
    def __len__(self):
        return 28539
    
    def __getitem__(self, index):
        return self.data[index]

In [9]:
import torch.nn.functional as F

autoencoder = AutoEncoder()
autoencoder.to(device)
print(autoencoder)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.0002)
EPOCH = 200
autoencoder.train()
train_dataset = Mydataset(data)
train_loader = DataLoader(train_dataset, batch_size=256, drop_last=True, shuffle=True, num_workers=2)   
for epoch in range(EPOCH):
    for batch_num, train_data in tqdm(enumerate(train_loader), total=len(train_loader)):
        data_in = Variable(add_noise(train_data.view(-1, 1, 64,64)).double())
        data_out = Variable(train_data.view(-1, 1, 64,64).float())
        #print(torch.from_numpy(data[step]).view(-1, 1, 64,64).double())
        data_in = data_in.to(torch.float32)
        data_out.to(device)
        encoded, decoded = autoencoder(data_in.to(device))
        loss = F.binary_cross_entropy(decoded, data_out.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss = loss.item()

AutoEncoder(
  (Encoder): Sequential(
    (0): Conv2d(1, 32, kernel_size=(9, 9), stride=(2, 2), padding=(4, 4))
    (1): LeakyReLU(negative_slope=0.01)
    (2): Conv2d(32, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (3): LeakyReLU(negative_slope=0.01)
    (4): Conv2d(64, 128, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
    (5): LeakyReLU(negative_slope=0.01)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=8192, out_features=256, bias=True)
  )
  (Decoder_part1): Sequential(
    (0): Linear(in_features=256, out_features=8192, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
  )
  (Decoder_part2): Sequential(
    (0): ConvTranspose2d(128, 128, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2), output_padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01)
    (2): ConvTranspose2d(128, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), output_padding=(1, 1))
    (3): LeakyReLU(negative_slope=0.01)
    (4): ConvTranspose2d(64, 32, kernel_si

100%|██████████| 111/111 [00:16<00:00,  6.67it/s]


In [10]:
torch.save(autoencoder, 'autoencoder.pkl')

In [11]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(256, 1024),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 1024),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 5),
            #nn.Softmax()
        )

    def forward(self, x):
        x = self.fc(x)
        return x

In [12]:
emotion_dict = {'ang': 0,
                'exc': 1,
                'sad': 2,
                'fru': 3,
                'neu': 4}
valid_emo=['ang', 'exc', 'sad', 'fru', 'neu']

In [13]:
import pickle
#audio_vectors = pickle.load(open('/kaggle/input/vector/audio_vectors_1.pkl', 'rb'))
audio_vectors_path = '/kaggle/input/vector/audio_vectors_'
labels_df = pd.read_csv('/kaggle/input/vector/df_iemocap.csv')

In [14]:
import librosa
import librosa.display
import random
from sklearn.preprocessing import normalize
sr = 44100
def extract_features(vedio):
    Spec = librosa.feature.melspectrogram(y=vedio, sr=sr, n_mels=64, hop_length=int(sr*16/1000), win_length=int(sr*32/1000))
    N, D = Spec.shape
    if D < 64:
        return Spec
    if D > 64:
        rand = random.randint(64, D-1)
        Spec = Spec[:, rand-64: rand]
    result = normalize(Spec, axis=1, norm='max')
    return result

In [15]:
#training data
#autoencoder = torch.load('/kaggle/input/autoencoders/Dautoencoder.pkl')
pre = torch.zeros((8000, 256))
emo = torch.zeros(8000)
cnt = 0
for sess in [2,3,4,5]:
    print("1")
    audio_vectors = pickle.load(open('{}{}.pkl'.format(audio_vectors_path, sess), 'rb'))
    for index, row in tqdm(labels_df[labels_df['wav_file'].str.contains('Ses0{}'.format(sess))].iterrows()):
        try:
            wave_file_name = row['wav_file']
            if row['emotion'] not in valid_emo:
                continue
            label = emotion_dict[row['emotion']]
            y = audio_vectors[wave_file_name]
            s = extract_features(y)
            N, D = s.shape
            if D < 64:
                continue
            data_in = Variable(torch.from_numpy(extract_features(y)).view(-1, 1, 64,64).double())
            data_in = data_in.to(torch.float32)
            with torch.no_grad():
                #print(autoencoder(data_in.to(device)))
                #print(autoencoder(data_in.to(device)))
                pre[cnt, :], _ = autoencoder(data_in.to(device))
                #pre[cnt, :], _ = autoencoder(data_in.to(device)).cpu().numpy()
            emo[cnt] = label
            cnt += 1
        except:
            print("one error occur")
#np.save(file='training.npy', arr=pre)
#np.save(file='label.npy', arr=emo)

1


1811it [00:37, 48.38it/s]


1


2136it [00:47, 44.81it/s]


1


2103it [00:42, 49.62it/s]


1


2170it [00:50, 42.87it/s]


In [16]:
torch.save(pre, "./training.pth")
torch.save(emo, "./label.pth")
del pre
del emo

In [17]:
training = torch.load('training.pth')
label = torch.load('label.pth')
label.shape

torch.Size([8000])

In [18]:
def to_one_hot(label, dimension=5):
    results = np.zeros(dimension)
    results[int(label)] = 1
    return results

In [19]:
class My2dataset(Dataset):
    def __init__(self, training, label):
        self.data = training
        self.label = label
    
    def __len__(self):
        return len(label)
    
    def __getitem__(self, index):
        return self.data[index], self.label[index]

In [20]:
import warnings
warnings.filterwarnings("ignore")

In [21]:
pre = torch.zeros((2000, 256))
emo = torch.zeros(2000)
valid_num = 0
correct_num = 0

In [22]:
def classified(vedio):
    Spec = librosa.feature.melspectrogram(y=vedio, sr=sr, n_mels=64, hop_length=int(sr*16/1000), win_length=int(sr*32/1000))
    N, D = Spec.shape
    proba = torch.zeros(5)
    end = 64
    cnt = 0
    while end < D:
        #print(end)
        temp = Spec[:, end-64: end]
        temp = normalize(temp, axis=1, norm='max')
        temp = Variable(torch.from_numpy(temp).view(-1, 1, 64, 64).double())
        temp = temp.to(torch.float32)
        with torch.no_grad():
            temp, _ = autoencoder(temp.to(device))
        #print(classifier(temp))
        #print(proba)
        temp = classifier(temp.to(device))
        soft = nn.Softmax()
        temp = soft(temp)
        proba = proba.to(device) + temp
        end += 32
        cnt += 1
    return torch.argmax(proba)

In [23]:
EPOCH = 50
audio_vectors = pickle.load(open('{}{}.pkl'.format(audio_vectors_path, 1), 'rb'))
classifier = Classifier()
print(classifier)
classifier.to(device)
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.0005)
loss_func = nn.CrossEntropyLoss()
train_dataset2 = My2dataset(training, label)
print(len(train_dataset2))
train_loader2 = DataLoader(train_dataset2, batch_size=256, drop_last=True, shuffle=True, num_workers=2)   
for epoch in range(EPOCH):
    for batch_num, (train_data, emotion) in tqdm(enumerate(train_loader2)):
        #print(batch_num)
        #print(train_data.shape)
        classifier.train()
        proba = classifier(train_data.to(device))
        proba = proba.view(-1, 5)
        y_target = emotion
        #print(proba)
        #print(y_target)
        loss = loss_func(proba.to(device), y_target.long().to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss = loss.item()
    with torch.no_grad():
        classifier.eval()
        pre = torch.zeros((2000, 256))
        emo = torch.zeros(2000)
        valid_num = 0
        correct_num = 0
        for sess in [1]:
            for index, row in tqdm(labels_df[labels_df['wav_file'].str.contains('Ses0{}'.format(sess))].iterrows()):
                wave_file_name = row['wav_file']
                if row['emotion'] not in valid_emo:
                    continue
                valid_num += 1
                #print(valid_num)
                emo = emotion_dict[row['emotion']]
                y = audio_vectors[wave_file_name]
                prediction = classified(y)
                if prediction == emo:
                    correct_num += 1
        accuracy = 1.0 * correct_num/valid_num
        print(accuracy)
            
    

Classifier(
  (fc): Sequential(
    (0): Linear(in_features=256, out_features=1024, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=1024, out_features=1024, bias=True)
    (4): LeakyReLU(negative_slope=0.01)
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=1024, out_features=5, bias=True)
  )
)
8000


31it [00:00, 86.32it/s] 
1819it [00:40, 44.52it/s]

0.35609756097560974



31it [00:00, 95.44it/s] 
1819it [00:40, 45.25it/s]

0.35853658536585364


...

0.33495934959349594



31it [00:00, 95.87it/s] 
1819it [00:40, 44.59it/s]

0.34146341463414637



31it [00:00, 97.52it/s] 
1819it [00:40, 45.03it/s]

0.3300813008130081


In [24]:
torch.save(classifier, 'emoclassifier.pkl')

test part

In [25]:
import warnings
warnings.filterwarnings("ignore")

In [26]:
def classified(vedio):
    Spec = librosa.feature.melspectrogram(y=vedio, sr=sr, n_mels=64, hop_length=int(sr*16/1000), win_length=int(sr*32/1000))
    N, D = Spec.shape
    proba = torch.zeros(5)
    end = 64
    cnt = 0
    while end < D:
        #print(end)
        temp = Spec[:, end-64: end]
        temp = normalize(temp, axis=1, norm='max')
        temp = Variable(torch.from_numpy(temp).view(-1, 1, 64, 64).double())
        temp = temp.to(torch.float32)
        with torch.no_grad():
            temp, _ = autoencoder(temp.to(device))
        #print(classifier(temp))
        #print(proba)
        temp = classifier(temp.to(device))
        soft = nn.Softmax()
        temp = soft(temp)
        proba = proba.to(device) + temp
        end += 32
        cnt += 1
    return torch.argmax(proba)

In [27]:
pre = torch.zeros((2000, 256))
emo = torch.zeros(2000)
valid_num = 0
correct_num = 0
#torch.load(classifier, 'emoclassifier.pkl')
for sess in [5]:
    audio_vectors = pickle.load(open('{}{}.pkl'.format(audio_vectors_path, sess), 'rb'))
    for index, row in tqdm(labels_df[labels_df['wav_file'].str.contains('Ses0{}'.format(sess))].iterrows()):
        wave_file_name = row['wav_file']
        if row['emotion'] not in valid_emo:
            continue
        valid_num += 1
        #print(valid_num)
        label = emotion_dict[row['emotion']]
        y = audio_vectors[wave_file_name]
        with torch.no_grad():
            prediction = classified(y)
        if prediction == label:
            correct_num += 1
accuracy = 1.0 * correct_num/valid_num
accuracy

2170it [00:49, 43.86it/s]


0.3881000676132522